In [1]:
!pip install -q transformers datasets sacrebleu sentencepiece accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.0 MB/s eta 0:00:00


In [2]:
import os, json, math, time
from datasets import load_dataset#, load_metric
from transformers import (
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)
import torch
import numpy as np

In [3]:
# replace the old load_cell
raw = load_dataset("wmt17", "zh-en",
                   split={"train":"train[:1%]",
                          "validation":"validation",
                          "test":"test"})
src, tgt = "en", "zh"
def flip(batch):
    batch["translation"] = {"en": batch["translation"]["en"],
                            "zh": batch["translation"]["zh"]}
    return batch
raw = raw.map(flip)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

zh-en/train-00000-of-00013.parquet:   0%|          | 0.00/286M [00:00<?, ?B/s]

zh-en/train-00001-of-00013.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

zh-en/train-00002-of-00013.parquet:   0%|          | 0.00/281M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/2002 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/251347 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

In [4]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tok = MBart50TokenizerFast.from_pretrained(model_name, src_lang="en_XX", tgt_lang="zh_CN")
model = MBartForConditionalGeneration.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [5]:
max_src, max_tgt = 128, 128
def encode(ex):
    en_sent = [item["en"] for item in ex["translation"]]
    zh_sent = [item["zh"] for item in ex["translation"]]
    # 一次调用同时编码源端和目标端
    model_inputs = tok(
        en_sent,
        text_target=zh_sent,
        max_length=max_src,
        truncation=True,
        padding=False,          # 动态 padding，由 data_collator 完成
    )
    return model_inputs

tokenised = raw.map(encode, batched=True, remove_columns=raw["train"].column_names)

Map:   0%|          | 0/251347 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

In [6]:
data_coll = DataCollatorForSeq2Seq(tok, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="mbart-en-zh-wmt14",
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=500,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=max_tgt,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

In [7]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [8]:
import evaluate

bleu = evaluate.load("sacrebleu")
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    pred_str = tok.batch_decode(preds, skip_special_tokens=True)
    label_str = tok.batch_decode(labels, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=pred_str, references=[[r] for r in label_str])["score"]
    return {"bleu": bleu_score}

In [9]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenised["train"],
    eval_dataset=tokenised["validation"],
    processing_class=tok,
    data_collator=data_coll,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [10]:
trainer.args

Seq2SeqTrainingArguments(output_dir='mbart-en-zh-wmt14', overwrite_output_dir=False, do_train=False, do_eval=True, do_predict=False, eval_strategy=<IntervalStrategy.STEPS: 'steps'>, prediction_loss_only=False, per_device_train_batch_size=4, per_device_eval_batch_size=4, per_gpu_train_batch_size=None, per_gpu_eval_batch_size=None, gradient_accumulation_steps=4, eval_accumulation_steps=None, eval_delay=0, torch_empty_cache_steps=None, learning_rate=3e-05, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=3, max_steps=-1, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs={}, warmup_ratio=0.0, warmup_steps=0, log_level='passive', log_level_replica='warning', log_on_each_node=True, logging_dir='mbart-en-zh-wmt14/runs/Nov04_09-07-21_17677297758b', logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_first_step=False, logging_steps=100, logging_nan_inf_filter=True, save_strategy=<SaveStrategy.STEPS: 'steps'

In [11]:
trainer.train()


Step,Training Loss,Validation Loss,Bleu
500,1.945200,2.243330,1.467716
1000,1.856000,2.240997,1.495561
1500,1.808400,2.231154,1.638939
2000,1.817300,2.237159,1.478944
2500,1.762000,2.228672,1.553495


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2500, training_loss=1.8545634765625, metrics={'train_runtime': 7916.9404, 'train_samples_per_second': 95.244, 'train_steps_per_second': 5.953, 'total_flos': 4315855863545856.0, 'train_loss': 1.8545634765625, 'epoch': 0.15914190683832774})

In [12]:
trainer.evaluate(tokenised["test"])

{'eval_loss': 2.2744877338409424,
 'eval_bleu': 4.304310131461501,
 'eval_runtime': 652.7008,
 'eval_samples_per_second': 3.066,
 'eval_steps_per_second': 0.768,
 'epoch': 0.15914190683832774}

In [13]:
def translate_en_zh(sentence: str) -> str:
    model.eval()
    with torch.no_grad():
        inputs = tok(sentence, return_tensors="pt").to(model.device)
        generated = model.generate(**inputs,
                                   forced_bos_token_id=tok.lang_code_to_id["zh_CN"],
                                   max_length=150,
                                   num_beams=5,
                                   early_stopping=True)
    return tok.batch_decode(generated, skip_special_tokens=True)[0]

translate_en_zh("Machine translation is not quite solved yet.")

'机器翻译还没有完全解决。'

In [14]:
trainer.save_model("mbart-en-zh-wmt14-best")
tok.save_pretrained("mbart-en-zh-wmt14-best")

('mbart-en-zh-wmt14-best/tokenizer_config.json',
 'mbart-en-zh-wmt14-best/special_tokens_map.json',
 'mbart-en-zh-wmt14-best/sentencepiece.bpe.model',
 'mbart-en-zh-wmt14-best/added_tokens.json',
 'mbart-en-zh-wmt14-best/tokenizer.json')